In [ ]:
# 01_setup_and_aoi.ipynb

# ---- Cell 1: Install libraries (run once per Colab session) ----
!pip install -q earthengine-api geemap geopandas

# ---- Cell 2: Authenticate and initialize Earth Engine ----
import ee

# Opens a browser login. Follow the prompts and paste the code back into Colab.
ee.Authenticate()

# Replace with your own Google Cloud project ID registered for Earth Engine
PROJECT_ID = 'terraworks-511519'
ee.Initialize(project=PROJECT_ID)

print('Earth Engine initialized:', ee.String('ready').getInfo())

# ---- Cell 3: Define the study area ----
# Approximate centre of Ujani Reservoir (Bhima river, Pune/Solapur border, Maharashtra)
UJANI_LAT = 18.0
UJANI_LON = 75.1

# Point of interest
centre = ee.Geometry.Point([UJANI_LON, UJANI_LAT])

# Study area: a buffer around the centre to capture the reservoir and nearby catchment.
# 10 km radius for context; tighten to 3 km if you only need the water body.
aoi = centre.buffer(10000).bounds()   # bounding box of a 10 km circle

# Optional: load a reservoir boundary you exported from QGIS or OpenStreetMap instead
# aoi = ee.FeatureCollection('users/your-username/ujani_aoi').geometry()

# Print the area in square kilometres as a sanity check
area_km2 = aoi.area(maxError=1).divide(1e6).getInfo()
print(f'Study area: {area_km2:.1f} km"')

# ---- Cell 4: Map the study area ----
import geemap

Map = geemap.Map(center=[UJANI_LAT, UJANI_LON], zoom=11)
Map.addLayer(aoi, {'color': 'red'}, 'Study area')
Map.addLayer(centre, {'color': 'blue'}, 'Reservoir centre (approx.)')
Map

# ---- Cell 5: Check that Sentinel-2 imagery exists for the area ----
# Use one recent dry-season window to confirm data availability
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2024-01-01', '2024-03-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

count = s2.size().getInfo()
print(f'Sentinel-2 images found (Jan–Mar 2024, <20% cloud): {count}')

# Show one true-colour image as a visual check
if count > 0:
    rgb_image = s2.median().select(['B4', 'B3', 'B2'])
    Map.addLayer(rgb_image, {'min': 0, 'max': 3000}, 'Sentinel-2 true colour (median)')
    print('True-colour layer added to the map.')
else:
    print('No images found. Widen the date range or raise the cloud threshold.')